In [1]:
!apt-get update -y -qq
!apt-get install -y -qq libgsl-dev build-essential

!git clone https://github.com/scwatts/fastspar.git

%cd fastspar
!./autogen.sh
!./configure
!make -j4

!./src/fastspar --version

print("FastSpar ready")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
fatal: destination path 'fastspar' already exists and is not an empty directory.
/content/fastspar
checking for a BSD-compatible install... /usr/bin/install -c
checking whether build environment is sane... yes
checking for a race-free mkdir -p... /usr/bin/mkdir -p
checking for gawk... no
checking for mawk... mawk
checking whether make sets $(MAKE)... yes
checking whether make supports nested variables... yes
checking for g++... g++
checking whether the C++ compiler works... yes
checking for C++ compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C++... yes
checking whether g++ accepts -g... yes
checking for g++ option to enable C++11 features.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE = "/content/drive/MyDrive"

cd_file = f"{BASE}/CD_network_counts_FIXED.tsv"
uc_file = f"{BASE}/UC_network_counts_FIXED.tsv"
h_file  = f"{BASE}/Healthy_network_counts_FIXED.tsv"

print("Files ready:")
print(cd_file, os.path.exists(cd_file))
print(uc_file, os.path.exists(uc_file))
print(h_file, os.path.exists(h_file))

Files ready:
/content/drive/MyDrive/CD_network_counts_FIXED.tsv True
/content/drive/MyDrive/UC_network_counts_FIXED.tsv True
/content/drive/MyDrive/Healthy_network_counts_FIXED.tsv True


In [4]:
import os
import shutil
import pandas as pd

print("Copying count files from Drive → Colab\n")

for cond in ["CD", "UC", "Healthy"]:
    src = f"/content/drive/MyDrive/{cond}_network_counts_FIXED.tsv"
    dst = f"/content/{cond}_network_counts_FIXED.tsv"

    if os.path.exists(src):
        shutil.copy(src, dst)

        df = pd.read_csv(dst, sep="\t", index_col=0)

        print(f"{cond}: copied successfully")
        print(f"   Shape: {df.shape}")
    else:
        print(f"{cond}: NOT FOUND")
        print(f"   Expected at: {src}")

Copying count files from Drive → Colab

CD: copied successfully
   Shape: (748, 87)
UC: copied successfully
   Shape: (456, 87)
Healthy: copied successfully
   Shape: (428, 87)


In [5]:
import pandas as pd

for cond in ["CD", "UC", "Healthy"]:
    path = f"/content/{cond}_network_counts_FIXED.tsv"

    df = pd.read_csv(path, sep="\t", index_col=0)

    df = df.T

    df.reset_index(inplace=True)

    df.rename(columns={"index": "#OTU ID"}, inplace=True)

    df.to_csv(path, sep="\t", index=False)

    print(f"Fixed format for {cond}: {df.shape}")

Fixed format for CD: (87, 749)
Fixed format for UC: (87, 457)
Fixed format for Healthy: (87, 429)


In [6]:
pd.read_csv("/content/CD_network_counts_FIXED.tsv", sep="\t").head()

,#OTU ID,CSM5MCVL,CSM5MCVN,CSM5MCW6,CSM5MCWC,CSM5MCWE,CSM5MCWG,CSM5MCXD,CSM5MCXT,CSM5MCXV,...,CSM5MCU4_P,MSM5LLHI_P,ESM5ME9D_P,MSM5LLHO_P,CSM5MCVJ_P,CSM5MCWI_P,MSM5LLHA_P,HSM5MD4A_P,ESM5MEBA_P,HSM5FZC2_P
0,Akkermansia_muciniphila,0,0,0,0,0,0,310,0,0,...,0,345,0,172,0,22,16,597,0,0
1,Alistipes_finegoldii,0,0,0,0,0,0,0,1,5,...,0,1095,0,614,0,0,13,0,0,0
2,Alistipes_onderdonkii,0,0,0,0,0,0,0,9,27,...,0,112,0,109,0,0,4,0,0,77
3,Alistipes_putredinis,0,0,0,1,0,0,0,795,861,...,0,628,0,1075,0,0,1,0,0,508
4,Alistipes_shahii,0,0,0,0,0,0,0,0,1,...,0,1617,0,469,0,0,45,0,0,0


In [7]:
import os, subprocess, glob, time

def run_full_fastspar(condition, n_bootstrap=100, iterations=20):

    counts = f"/content/{condition}_network_counts_FIXED.tsv"

    base_out = f"/content/fastspar/sparcc_{condition}"
    boot_counts = f"{base_out}/bootstrap_counts"
    boot_cors = f"{base_out}/bootstrap_cors"
    boot_only = f"{base_out}/bootstrap_corr_only"

    os.makedirs(base_out, exist_ok=True)
    os.makedirs(boot_counts, exist_ok=True)
    os.makedirs(boot_cors, exist_ok=True)
    os.makedirs(boot_only, exist_ok=True)

    print(f"Processing: {condition}")

    print("Step 1/4: Main correlations...")

    r = subprocess.run([
        "./src/fastspar",
        "--otu_table", counts,
        "--correlation", f"{base_out}/cor_{condition}.csv",
        "--covariance", f"{base_out}/cov_{condition}.csv",
        "--iterations", str(iterations),
        "--yes"
    ], cwd="/content/fastspar", capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR in main correlation:")
        print(r.stderr[:300])
        return False

    print("Done")

    print(f"Step 2/4: Generating {n_bootstrap} bootstraps...")

    subprocess.run("rm -f /content/fastspar/boot_*.tsv", shell=True)

    r = subprocess.run([
        "./src/fastspar_bootstrap",
        "--otu_table", counts,
        "--number", str(n_bootstrap),
        "--prefix", "boot_"
    ], cwd="/content/fastspar", capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR in bootstrap generation:")
        print(r.stderr[:300])
        return False

    boot_files = glob.glob("/content/fastspar/boot_*.tsv")

    if len(boot_files) == 0:
        print("CRITICAL: No bootstrap files generated!")
        return False

    for f in boot_files:
        subprocess.run(["mv", f, boot_counts])

    print(f"{len(boot_files)} bootstrap files created")

    boot_files = sorted(glob.glob(f"{boot_counts}/boot_*.tsv"))

    print(f"Step 3/4: Running bootstrap correlations ({len(boot_files)})...")

    for i, bf in enumerate(boot_files):
        base = os.path.basename(bf).replace(".tsv", "")

        subprocess.run([
            "./src/fastspar",
            "--otu_table", bf,
            "--correlation", f"{boot_cors}/{base}.tsv",
            "--covariance", f"{boot_cors}/{base}_cov.tsv",
            "--iterations", str(iterations),
            "--yes"
        ], cwd="/content/fastspar", capture_output=True)

        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(boot_files)}")

    print("Bootstrap correlations done")


    for f in glob.glob(f"{boot_cors}/*.tsv"):
        if "_cov" not in f:
            subprocess.run(["cp", f, boot_only])

    print(f"{len(os.listdir(boot_only))} correlation files ready")

    print("Step 4/4: Computing p-values...")

    r = subprocess.run([
        "./src/fastspar_pvalues",
        "--otu_table", counts,
        "--correlation", f"sparcc_{condition}/cor_{condition}.csv",
        "--prefix", f"sparcc_{condition}/bootstrap_corr_only/boot_",
        "--permutations", str(n_bootstrap),
        "--outfile", f"sparcc_{condition}/pvals_{condition}.csv"
    ], cwd="/content/fastspar", capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR in p-values:")
        print(r.stderr[:300])
        return False

    print("P-values computed successfully")

    return True


start = time.time()

for cond in ["CD", "UC", "Healthy"]:
    success = run_full_fastspar(cond, n_bootstrap=100, iterations=20)

    if success:
        print(f"{cond} COMPLETE")
    else:
        print(f"{cond} FAILED")

print(f"\nTotal time: {(time.time() - start)/60:.1f} minutes")

Processing: CD
Step 1/4: Main correlations...
Done
Step 2/4: Generating 100 bootstraps...
100 bootstrap files created
Step 3/4: Running bootstrap correlations (100)...
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 100/100
Bootstrap correlations done
100 correlation files ready
Step 4/4: Computing p-values...
P-values computed successfully
CD COMPLETE
Processing: UC
Step 1/4: Main correlations...
Done
Step 2/4: Generating 100 bootstraps...
100 bootstrap files created
Step 3/4: Running bootstrap correlations (100)...
  Progress: 10/100
  Progress: 20/100
  Progress: 30/100
  Progress: 40/100
  Progress: 50/100
  Progress: 60/100
  Progress: 70/100
  Progress: 80/100
  Progress: 90/100
  Progress: 100/100
Bootstrap correlations done
100 correlation files ready
Step 4/4: Computing p-values...
P-values computed successfully
UC COMPLETE
Processing: Healthy


In [8]:
import os

for cond in ["CD", "UC", "Healthy"]:
    path = f"/content/fastspar/sparcc_{cond}/bootstrap_counts"
    print(f"\n{cond}:")
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"Files found: {len(files)}")
        print(files[:5])
    else:
        print("Folder not found")


CD:
Files found: 100
['boot__86.tsv', 'boot__23.tsv', 'boot__58.tsv', 'boot__90.tsv', 'boot__6.tsv']

UC:
Files found: 100
['boot__86.tsv', 'boot__23.tsv', 'boot__58.tsv', 'boot__90.tsv', 'boot__6.tsv']

Healthy:
Files found: 100
['boot__86.tsv', 'boot__23.tsv', 'boot__58.tsv', 'boot__90.tsv', 'boot__6.tsv']


In [9]:
import os

for cond in ["CD", "UC", "Healthy"]:
    path = f"/content/fastspar/sparcc_{cond}"
    print(f"\n{cond}:")
    print(os.listdir(path))


CD:
['cor_CD.csv', 'bootstrap_counts', 'bootstrap_cors', 'cov_CD.csv', 'pvals_CD.csv', 'bootstrap_corr_only']

UC:
['bootstrap_counts', 'bootstrap_cors', 'cov_UC.csv', 'pvals_UC.csv', 'cor_UC.csv', 'bootstrap_corr_only']

Healthy:
['pvals_Healthy.csv', 'cor_Healthy.csv', 'cov_Healthy.csv', 'bootstrap_counts', 'bootstrap_cors', 'bootstrap_corr_only']


In [10]:
print("VERIFICATION:")

for cond in ["CD", "UC", "Healthy"]:
    cor_path = f"/content/fastspar/sparcc_{cond}/cor_{cond}.csv"
    pval_path = f"/content/fastspar/sparcc_{cond}/pvals_{cond}.csv"

    if os.path.exists(cor_path) and os.path.exists(pval_path):
        cor = pd.read_csv(cor_path, sep=None, engine='python', index_col=0)
        pval = pd.read_csv(pval_path, sep=None, engine='python', index_col=0)

        pval = pval.apply(pd.to_numeric, errors='coerce').fillna(1.0)

        print(f"\nDEBUG {cond}: shape = {cor.shape}")

        sig = ((cor.abs() > 0.3) & (pval < 0.05))
        n_sig = sig.values.sum() // 2

        print(f"{cond}:")
        print(f"  Matrix shape: {cor.shape}")
        print(f"  Significant edges: {n_sig}")

    else:
        print(f" {cond}: files missing")

VERIFICATION:

DEBUG CD: shape = (87, 87)
CD:
  Matrix shape: (87, 87)
  Significant edges: 101

DEBUG UC: shape = (87, 87)
UC:
  Matrix shape: (87, 87)
  Significant edges: 95

DEBUG Healthy: shape = (87, 87)
Healthy:
  Matrix shape: (87, 87)
  Significant edges: 184


In [11]:
import os
import shutil

config = {
    "CD": ("sparcc_CD", "CD"),
    "UC": ("sparcc_UC", "UC"),
    "Healthy": ("sparcc_Healthy", "Healthy")
}

os.makedirs("/content/sparcc_results", exist_ok=True)

for cond, (folder, fname) in config.items():
    base = f"/content/fastspar/{folder}"

    cor_file = f"{base}/cor_{fname}.csv"
    pval_file = f"{base}/pvals_{fname}.csv"

    if os.path.exists(cor_file):
        shutil.copy(cor_file, "/content/sparcc_results/")
        print(f"cor_{fname}.csv")
    else:
        print(f"Missing: {cor_file}")

    if os.path.exists(pval_file):
        shutil.copy(pval_file, "/content/sparcc_results/")
        print(f"pvals_{fname}.csv")
    else:
        print(f"Missing: {pval_file}")

shutil.make_archive("/content/sparcc_final", "zip", "/content/sparcc_results")


cor_CD.csv
pvals_CD.csv
cor_UC.csv
pvals_UC.csv
cor_Healthy.csv
pvals_Healthy.csv


'/content/sparcc_final.zip'

In [12]:
import os

for cond in ["sparcc_CD", "sparcc_UC", "sparcc_Healthy"]:
    print("\n", cond)
    print(os.listdir(f"/content/fastspar/{cond}"))


 sparcc_CD
['cor_CD.csv', 'bootstrap_counts', 'bootstrap_cors', 'cov_CD.csv', 'pvals_CD.csv', 'bootstrap_corr_only']

 sparcc_UC
['bootstrap_counts', 'bootstrap_cors', 'cov_UC.csv', 'pvals_UC.csv', 'cor_UC.csv', 'bootstrap_corr_only']

 sparcc_Healthy
['pvals_Healthy.csv', 'cor_Healthy.csv', 'cov_Healthy.csv', 'bootstrap_counts', 'bootstrap_cors', 'bootstrap_corr_only']
